<h1>Chapter 5 - Tools</h1>
<i>Giving an Agent access to the Environment through Tool Usage</i>


<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents/blob/main/chapter05/chapter05_native_tool_calling.ipynb)

---

This notebook is for Chapter 5 of [An Illustrated Guide to AI Agents](https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ) by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="https://www.amazon.com/Illustrated-Guide-AI-Agents-Concepts/dp/B0GTYL2QSJ">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM - Gemma 4

At the beginning of every chapter, we start by choosing the LLM that we want to use. In this notebook, we will explore how tool calling works with a model that already has this capabilities, no need for prompting! As such, the model that we will be using throughout this chapter is Gemma 4.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B (with native thinking and tool calling)
llm = LLM(model="gemma4:e4b")

If you want to use another LLM, here are a couple of options (both locally and on the cloud) that you can try:

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2 - *"Magically"* Creating Tool Calls

In `chapter05.ipynb`, we covered the basic steps of tool creation, definition, selection, calling, and output processing. With native tool calling, the LLM itself is learned during training to do much of that by itself. Although the steps remain the same, the way you approach them is mostly simplified.

![../images/ch5_tools.png](../images/ch5_tools.png)

To understand what is happening under the hood, let's go back to interacting with an `openai` endpoint to see how tool calls are formatted. To do so, let's start again with creating our tools and what it means to define a tool for an LLM that has native tool calling.

We can use the same tools we had before:



In [ ]:
def multiply(a: str, b: str) -> float:
    """Multiply two values."""
    return float(a) * float(b)

def get_weather(location: str) -> str:
    """Get the weather using a `location` parameter."""
    return f"Weather in {location}: Sunny, 72°F"

However, defining them and passing them along to the LLM works quite a bit different compared to prompting. `openai`-compatible end-points require a very specific formatting in JSON to describe the tools that you have. For instance, our `get_weather` tool would require a format like so:

In [ ]:
tool_schema = {
  "type": "function",
  "function": {
    "name": "multiply",
    "description": "Multiply two numbers",
    "parameters": {
      "type": "object",
      "properties": {
        "a": {
          "type": "number",
          "description": "First number"
        },
        "b": {
          "type": "number",
          "description": "Second number"
        }
      },
      "required": [
        "a",
        "b"
      ]
    }
  }
}

This JSON format is what the `openai`-endpoint expects and gets converted into the appropriate chat template the model is using. We can then pass this along to the `openai` endpoint which automatically converts and parses this:

In [ ]:
import json
import urllib.request
from rich import print

# Prompt
messages = [{"role": "user", "content": "What is 5.1 times 7.3?"}]

# Prepare the data
body = {
    "model": "gemma4:e4b",
    "messages": messages,
    "stream": False,
    "tools": [tool_schema],
}

# Prepare the request
request = urllib.request.Request(
    "http://localhost:11434/api/chat",
    data=json.dumps(body).encode(),
    headers={"Content-Type": "application/json"},
)

# Call and parse the response
with urllib.request.urlopen(request) as response:
    print(json.loads(response.read())["message"])

However, this doesn't quite explain how everything works now does it? Seeing the above, it still feels like magic (at least to us)! 

> **How** does it update the prompt? **How** is it converted to the chat template? **How** does the model then call a tool? Etc.

In our previous example in `chapter05.ipynb`, we told the model what kind of schema it could expect (i.e., JSON) and we had to parse it ourselves. Now, the model already has a fixed way of doing this and the endpoint is in charge of then converting it back.

Let's see how this is done!

## 3. Uncovering the *"Magic"*

Under the hood, the LLM is expecting a very specific prompt template. For Gemma 4, it roughly expects the following:

<pre style="background:#1e1e1e; color:#d4d4d4; padding:16px; border-radius:8px; overflow-x:auto; font-size:13px; line-height:1.8;">
<span style="color:#f97583;">&lt;|turn&gt;system</span>
<span style="color:#79b8ff;">&lt;|tool&gt;</span><span style="color:#85e89d;">declaration:</span><span style="background:#85e89d22; color:#85e89d; padding:1px 6px; border-radius:3px;">FUNCTION_NAME</span>{description:<span style="color:#ffab70;">&lt;|"|&gt;</span><span style="background:#ffab7022; color:#ffab70; padding:1px 6px; border-radius:3px;">DESCRIPTION</span><span style="color:#ffab70;">&lt;|"|&gt;</span>,parameters:{properties:{<span style="background:#85e89d22; color:#85e89d; padding:1px 6px; border-radius:3px;">PARAM_NAME</span>:{description:<span style="color:#ffab70;">&lt;|"|&gt;</span><span style="background:#ffab7022; color:#ffab70; padding:1px 6px; border-radius:3px;">PARAM_DESC</span><span style="color:#ffab70;">&lt;|"|&gt;</span>,type:<span style="color:#ffab70;">&lt;|"|&gt;</span><span style="background:#ffab7022; color:#ffab70; padding:1px 6px; border-radius:3px;">TYPE</span><span style="color:#ffab70;">&lt;|"|&gt;</span>}},required:[<span style="color:#ffab70;">&lt;|"|&gt;</span><span style="background:#ffab7022; color:#ffab70; padding:1px 6px; border-radius:3px;">PARAM_NAME</span><span style="color:#ffab70;">&lt;|"|&gt;</span>],type:<span style="color:#ffab70;">&lt;|"|&gt;</span>OBJECT<span style="color:#ffab70;">&lt;|"|&gt;</span>}}<span style="color:#79b8ff;">&lt;tool|&gt;</span><span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;user</span>
<span style="background:#D7D7D7; color:#000000; padding:1px 6px; border-radius:3px;">USER_PROMPT</span><span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;model</span>
</pre>

You can see that the system prompt is filled with information about the tool. Instead of having to create the system prompt ourselves, it is populated through the prompt template of the model. The reason why this is so important is that the model was trained specifically for tool calling using this template. If we were to diverge from that the model might have a difficult time. 

So how does the following JSON get converted to what the model expects?


```json
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get the weather using a location parameter.",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string",
          "description": "The city name"
        }
      },
      "required": ["location"]
    }
  }
}
```

In practice, the underlying provider (`ollama`) parses that JSON input and converts it to what the specific LLM expects:

| JSON input | | Gemma 4 output |
|---|---|---|
| `"name": "get_weather"` | → | `declaration:get_weather{...}` |
| `"description": "Get the..."` | → | `description:<\|"\|>Get the...<\|"\|>` |
| `"type": "string"` | → | `type:<\|"\|>STRING<\|"\|>` |
| `"required": ["location"]` | → | `required:[<\|"\|>location<\|"\|>]` |
| JSON quotes `"..."` | → | Gemma quotes `<\|"\|>...<\|"\|>` |

Which gives us the following updated prompt:

<pre style="background:#1e1e1e; color:#d4d4d4; padding:16px; border-radius:8px; overflow-x:auto; font-size:13px; line-height:1.6;">
<span style="color:#f97583;">&lt;|turn&gt;system</span>
<span style="color:#79b8ff;">&lt;|tool&gt;</span><span style="color:#85e89d;">declaration:get_weather</span>{description:<span style="color:#ffab70;">&lt;|"|&gt;</span>Get the weather using a location parameter.<span style="color:#ffab70;">&lt;|"|&gt;</span>,parameters:{properties:{location:{description:<span style="color:#ffab70;">&lt;|"|&gt;</span>The city name<span style="color:#ffab70;">&lt;|"|&gt;</span>,type:<span style="color:#ffab70;">&lt;|"|&gt;</span>STRING<span style="color:#ffab70;">&lt;|"|&gt;</span>}},required:[<span style="color:#ffab70;">&lt;|"|&gt;</span>location<span style="color:#ffab70;">&lt;|"|&gt;</span>],type:<span style="color:#ffab70;">&lt;|"|&gt;</span>OBJECT<span style="color:#ffab70;">&lt;|"|&gt;</span>}}<span style="color:#79b8ff;">&lt;tool|&gt;</span><span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;user</span>
What is the weather in Paris?<span style="color:#f97583;">&lt;turn|&gt;</span>
<span style="color:#f97583;">&lt;|turn&gt;model</span>
</pre>


Let's see if we can try that out ourselves by querying the model directly rather than through the parsing of `ollama`. We first define the above prompt ourselves:

In [ ]:
# We parsed the chat template ourselves into what the model expects
raw_prompt = """<|turn>system
<|tool>declaration:multiply{description:<|"|>Multiply two values `a` and `b`.<|"|>,parameters:{properties:{a:{type:<|"|>STRING<|"|>},b:{type:<|"|>STRING<|"|>}},required:[<|"|>a<|"|>,<|"|>b<|"|>],type:<|"|>OBJECT<|"|>}}<tool|><turn|>
<|turn>user
What is 5.1 multiplied by 7.3?<turn|>
<|turn>model
"""

Then, we can pass this prompt directly to the `api/generate` endpoint of `ollama` which bypasses any template processing:

In [ ]:
import urllib.request
import json

# Create data
data = json.dumps(
    {
        "model": llm.model,
        "prompt": raw_prompt,
        "raw": True,
        "stream": False
    }
).encode()

# Send it without any template processing
req = urllib.request.Request(
    "http://localhost:11434/api/generate",
    data=data,
    headers={"Content-Type": "application/json"}
)

# Get raw output
with urllib.request.urlopen(req) as response:
    print(json.loads(response.read()))

[Gemma 4's chat template](https://huggingface.co/google/gemma-4-E4B-it/blob/main/chat_template.jinja) has very specific instructions on how to format the prompts and by following those, each provider (like `ollama` or `llama.cpp`) has their own implementations of handling that chat template.

## 4. Creating Function Definitions

Now that you understand how it works under the hood, let's explore the first step in native tool calling, defining the template. As you saw before, there is still JSON schema that the `openai` endpoint expects. This standardization is necessary to easily convert it to a given model's chat template. We can create a small helper script that converts certain types and inspects the signature of the function using `inspect`. To illustrate:



In [ ]:
import inspect

# Extract metadata
name = get_weather.__name__  # Name of a function
parameters = inspect.signature(get_weather).parameters  # Parameters of a function
docstring = inspect.getdoc(get_weather)  # Docstring of a function

# Show metadata
print(f"NAME:\n{name}")
print(f"PARAMETERS:\n{parameters}")
print(f"DOCSTRING:\n{docstring}")

Using that we can create the small helper script:

In [ ]:
import inspect
from typing import Callable

# Convert specific types to string descriptions
TYPE_MAP = {
    str: "string", int: "integer", float: "number", 
    bool: "boolean", list: "array", dict: "object"
}

def tool_to_schema(function: Callable) -> dict:
    """Convert a Python function to an OpenAI-style tool schema."""
    signature = inspect.signature(function)

    # Extract meatadata
    properties, required = {}, []
    for name, parameter in signature.parameters.items():
        properties[name] = {"type": TYPE_MAP.get(parameter.annotation, "string")}
        if parameter.default is inspect.Parameter.empty:
            required.append(name)

    # Fill schema
    schema = {
        "type": "function",
        "function": {
            "name": function.__name__,
            "description": inspect.getdoc(function),
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required,
            },
        },
    }

    return schema

Usage of the script is fortunately straightforward and gives us the exact same schema as we used before:

In [ ]:
print(tool_to_schema(multiply))

## 5. Creating `NativeTools`

Now that we have a procedure for converting the tool calls, we can use the `tools` parameter in the `LLM` class that we explored in chapter 2. Next, let's create the new `NativeTools` class that manages our tools: 

In [ ]:
import json
from illustrated_agents.chapters.ch2 import Response
from illustrated_agents.chapters.ch5 import Tools


class NativeTools(Tools):
    """Tool registry using native function calling."""

    @property
    def schemas(self) -> list[dict]:
        """Return tool functions for native function calling."""
        return [
            tool_to_schema(tool["function"]) for tool in self.registry.values()
        ]

    @property
    def prompt(self) -> str:
        """Empty because we don't need a prompt for native tool calling"""
        return ""

    def parse(self, response: Response) -> Response:
        """Parse a tool call."""
        # If there's no tool call, return the response as is
        if not response.tool_call:
            return response

        # Extract the tool name and arguments from the tool call
        args = response.tool_call["function"]["arguments"]
        if isinstance(args, str):
            args = json.loads(args)
        tool_call = {
            "tool": response.tool_call["function"]["name"],
            "kwargs": args,
        }

        # Add the parsed tool call to the response
        return Response(
            content=response.content,
            reasoning=response.reasoning,
            tool_call=tool_call,
        )

    def observation(self, result: str) -> tuple[str, str]:
        """Native tool results use the 'tool' role."""
        return "tool", str(result)

    def is_done(self, response: Response) -> bool:
        """No tool call means the `TinyAgent` is done."""
        return not response.tool_call

When can then add the Tools, including their descriptions as follows:

In [ ]:
# Register tools
tools = NativeTools()
tools.add_tool("multiply", multiply)
tools.add_tool("get_weather", get_weather)

Then, we can use the `schemas` function to convert any tool that we have defined to a JSON schema:

In [ ]:
tools.schemas

And that's all that is needed for our `NativeTools`. We don't need `tools.prompt` since all descriptions are parsed by giving them as JSON to the chat template.

## 6 - Running `TinyAgent`

The `TinyAgent` can now be initialized with the newly created `NativeTools`:

In [ ]:
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import TinyAgent

# Tools
tools = NativeTools()
tools.add_tool("multiply", multiply)
tools.add_tool("get_weather", get_weather)

# Memory
memory = Memory()

# Initialize Agent
agent = TinyAgent(llm=llm, memory=memory, tools=tools)

Let's check if the `TinyAgent` uses a tool when confronted with a question that may require one.

In [ ]:
agent.run("What is 5.1 times 7.3?")

It sure did! Note that not all models will strictly follow instructions and will fail every so often. Either way, the prerendered output shows that a tool was correctly used!

Let's explore the traces of the interaction in case something went wrong:

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

We can also check if your `TinyAgent` will answer questions that do not require tool usage:

In [ ]:
agent.run("Hi! Tell me something about flamingos in two sentences.")

It does! 😁 LLMs these days (Jan. 2026) are great in deciding themselves which tool to use and when. However, that does not mean it is not fallible. Describing hundreds of tools will likely fill up the context window too much and make it difficult for the LLM to decide if to use a tool and which one. 

In [ ]:
from illustrated_agents.utils import TrajectoryViewer
TrajectoryViewer(agent.trajectory)

As always, let's end with the Agent's memory:

In [ ]:
from rich import print
print(agent.memory.get_messages())

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how we could natively call tools without the need for custom JSON and XML parsing instructions. To enable both native and non-native capabilities in your `TinyAgent` we needed to update various things, like `LLM`, `Memory`, and `TinyAgent`. We also added the `NativeTools` class which handled the output of native tool calling.

In [ ]:
from illustrated_agents.chapters.ch5 import what_we_built_native; what_we_built_native

# What's Next

You can now decide how to move forward! There are a number of notebooks that you can choose from before continuing to the next chapter:

* `chapter05_skills.ipynb` -- Covers how to use skills

In the next chapter, we will finally introduce the one powerful mechanism that converts the LLM into an Agent... the **for loop**! (Reason and Act to be specific 😉)